In [ ]:
from scipy.io import loadmat
from scipy.signal import filtfilt, find_peaks, firwin, medfilt
import numpy as np
import pandas as pd
import sqlite3
import os
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
conn = sqlite3.connect(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")
sql = """
SELECT r.Animal_Id, r.Cell_Id, r.Folderpath, r.Condition, r.exp_type
FROM Recordings as r
WHERE exp_type = 'juxta'
AND r.Condition = 'ramp'
AND USE = 1
ORDER BY r.Cell_Id DESC"""

datarow = pd.read_sql_query(sql, conn)
data_row = datarow.head(1)
conn.close()

In [ ]:
# def tonestim_output_ramp(data_row):
"""
Port of tonestim_output_ramp.m (ramp condition), returning a
1-row pandas DataFrame. Uses the schema helpers so all columns
exist even if certain data (like pupil) are missing.
"""

# constants
tone_letter = ['a', 'w', 'e', 't']
pupil_sr = 50
halftime_spikes = 8        # sec window for TriggRaster
rasterBins = halftime_spikes * 800
halftime_mot = 8           # sec around trigger for pupil/whisk
smooth_pupil_value = 15    # for smoothing pupil signal
mot_samples = halftime_mot * 2 * pupil_sr

print(f"{'':-^100}")
print(f" Animal_Id {data_row['Animal_Id'].iloc[0]}; Cell_Id {int(data_row['Cell_Id'].iloc[0])} being processed ")
print(f"{'':-^100}")

# init shared vars
tone_onset = None
tone_code = None
pupil_psth = None
whisk_psth = None
RasterTimes = None
RasterRows = None
RasterRate = None
all_whisk = None
mot_avg = None
trigger_time = None

########################################################
# load trigger/ephys info for juxta OR behav experiment
########################################################
if data_row['exp_type'].values == 'juxta':
# load exp_data.mat
    try:
        d_path = Path(data_row['Folderpath'].values[0], 'exp_data.mat')
        print(d_path)
        exp = loadmat(
            d_path,
            struct_as_record=False,
            squeeze_me=False,
            simplify_cells= True  
        )
    except Exception:
        print(f"Missing exp_data.mat for {data_row['Animal_Id']} Cell_Id {data_row['Cell_Id']}")
        # we'll still build an empty row at the end
        spkT = None
    else:
        processed_data = exp.get('processed_data', None)
        raw_data = exp.get('raw_data', None)
        exp_info = exp.get('info', None)
        analysis = exp.get('analysis', None)

    if 'spike_sorting_data' not in processed_data:
        KeyError(format('Cell_id {} processed_data empty', data_row.Cell_Id.values[0]))

    ## processing
    # spike times
    spkT = processed_data['spike_sorting_data']['spike_times']
    
    # get tone times
    tone_times = raw_data['ephys_data']['keyboard_times'][:]
    # get tone codes
    tone_codes = raw_data['ephys_data']['keyboard_codes'][:,0]

    # sampling rate
    ephys_sr = round(raw_data['ephys_data']['sampling_rate'])

    # get time calls in ephys samples
    tone_onset = np.zeros(len(tone_times))
    for idx, _ in enumerate(tone_times):
        tmp = np.abs(raw_data['ephys_data']['ephy_times'] - tone_times[idx])
        min_idx = np.argmin(tmp)
        tone_onset[idx] = min_idx

    # obtain psth information
    for idx, char in enumerate(tone_letter):
        this_code = ord(char) # translate character to ascii
        this_codes = np.flatnonzero(this_code == tone_codes) #find occurrences of ascii character in all stimulations

        tone_triggers = tone_onset[this_codes] # filter for code trigger timings in samples

        # do it manually





In [ ]:
df = pd.read_csv(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\ADN\FH8Soso\analysis\test.csv",
                 names=['truemat'])

In [ ]:
matatrue = df.truemat.values
py = raster['raster_times']

diffs = np.subtract(matatrue, py)
